In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Configurar la mitigación de errores

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



> **Note:** La versión beta de un nuevo modelo de ejecución ya está disponible. El modelo de ejecución dirigida ofrece mayor flexibilidad para personalizar tu flujo de trabajo de mitigación de errores. Consulta la guía del [Modelo de ejecución dirigida](/guides/directed-execution-model) para más información.


<details>
<summary><b>Versiones de paquetes</b></summary>

El código de esta página fue desarrollado con los siguientes requisitos.
Se recomienda usar estas versiones o más recientes.

```
qiskit-ibm-runtime~=0.43.1
```
</details>

Las técnicas de mitigación de errores permiten a los usuarios reducir los errores de los circuitos
modelando el ruido del dispositivo en el momento de la ejecución. Esto generalmente
supone una sobrecarga de preprocesamiento cuántico relacionada con el entrenamiento del modelo y
una sobrecarga de posprocesamiento clásico para mitigar los errores en los resultados
brutos usando el modelo generado.

El primitivo Estimator admite varias técnicas de mitigación de errores, incluyendo [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation), [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne), [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec) y [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). Consulta [Técnicas de mitigación y supresión de errores](error-mitigation-and-suppression-techniques) para obtener una explicación de cada una. Al usar los primitivos, puedes activar o desactivar métodos individuales. Consulta la sección [Configuración personalizada de errores](#advanced-error) para más detalles.

> **Note:** Sampler no admite la mitigación de errores, pero puedes usar el paquete [mthree](https://qiskit.github.io/qiskit-addon-mthree/) (mitigación de medición sin matrices) para realizar la mitigación de errores de forma local.

Estimator también admite `resilience_level`. El nivel de resiliencia especifica cuánta resiliencia construir frente a
los errores. Los niveles más altos generan resultados más precisos, a expensas de
tiempos de procesamiento más largos. Los niveles de resiliencia se pueden usar para configurar
la compensación entre costo y precisión al aplicar la mitigación de errores a tu consulta
de primitivos. La mitigación de errores reduce los errores (sesgo) en los resultados
procesando las salidas de una colección, o conjunto, de circuitos relacionados. El
grado de reducción de errores depende del método aplicado. El nivel de resiliencia
abstrae la elección detallada del método de mitigación de errores para permitir
a los usuarios razonar sobre la compensación entre costo y precisión adecuada para
su aplicación.

Dado esto, cada nivel corresponde a un método o métodos con
mayor nivel de sobrecarga de muestreo cuántico, lo que te permite experimentar
con diferentes compensaciones entre tiempo y precisión. La siguiente tabla muestra
qué niveles y métodos correspondientes están disponibles para cada uno de los
primitivos.

> **Info:** La mitigación de errores es específica de la tarea, por lo que las técnicas que puedes
> aplicar varían según si estás muestreando una distribución o generando
> valores esperados.

<span id="resilience-table"></span>

Estimator admite los siguientes niveles de resiliencia. Sampler no admite niveles de resiliencia.

| Nivel de resiliencia | Definición                                                                                                          | Técnica                                                                       |
|----------------------|---------------------------------------------------------------------------------------------------------------------|-------------------------------------------------------------------------------|
| 0                    | Sin mitigación                                                                                                      | Ninguna                                                                       |
| 1 [Predeterminado]   | Costos mínimos de mitigación: mitiga el error asociado con errores de lectura                                       | Twirled Readout Error eXtinction (TREX) y twirling de medición                |
| 2                    | Costos de mitigación medios. Generalmente reduce el sesgo en los estimadores, pero no garantiza un resultado sin sesgo. | Nivel 1 + Extrapolación de Ruido Cero (ZNE) y twirling de puertas

> **Info:** Los niveles de resiliencia están actualmente en versión beta, por lo que la sobrecarga de muestreo y
> la calidad de la solución variarán de circuito a circuito. Nuevas funciones,
> opciones avanzadas y herramientas de gestión se publicarán de forma continua.
> No se garantiza que se apliquen métodos específicos de mitigación de errores
> en cada nivel de resiliencia.

## Configurar Estimator con niveles de resiliencia
Puedes usar los niveles de resiliencia para especificar técnicas de mitigación de errores, o puedes configurar técnicas personalizadas individualmente como se describe en [Configuración personalizada de errores.](#advanced-error)

<details>
<summary>Nivel de resiliencia 0</summary>

No se aplica ninguna mitigación de errores al programa del usuario.

</details>

<details>
<summary>Nivel de resiliencia 1</summary>

El nivel 1 aplica **mitigación de errores de lectura** y **twirling de medición** mediante una técnica sin modelo conocida
como Twirled Readout Error eXtinction (TREX). Reduce el error de medición
diagonalizando el canal de ruido asociado a la medición mediante la
aplicación aleatoria de volteos de qubits a través de puertas X inmediatamente antes de la medición. Un
término de reescalado del canal de ruido diagonal se aprende mediante la
evaluación comparativa de circuitos aleatorios inicializados en el estado cero. Esto permite
al servicio eliminar el sesgo de los valores esperados que resultan del
ruido de lectura. Este enfoque se describe con más detalle en [Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738).

</details>

<details>
<summary>Nivel de resiliencia 2</summary>

El nivel 2 aplica las **técnicas de mitigación de errores incluidas en el nivel 1** y también aplica **twirling de puertas** y usa el **método de Extrapolación de Ruido Cero (ZNE)**. ZNE calcula un
valor esperado del observable para distintos factores de ruido
(etapa de amplificación) y luego usa los valores esperados medidos para
inferir el valor esperado ideal en el límite de ruido cero (etapa de extrapolación).
Este enfoque tiende a reducir los errores en los valores esperados, pero
no garantiza producir un resultado sin sesgo.

![Esta imagen muestra una gráfica. El eje x está etiquetado como Factor de amplificación de ruido. El eje y está etiquetado como Valor esperado. Una línea con pendiente positiva está etiquetada como Valor mitigado. Los puntos cerca de la línea son valores amplificados por ruido. Hay una línea horizontal justo por encima del eje X etiquetada como Valor exacto.](../docs/images/guides/configure-error-mitigation/resilience-2.svg "Ilustración del método ZNE")

La sobrecarga de este método escala con el número de factores de ruido. La
configuración predeterminada muestrea el valor esperado en tres factores de ruido,
lo que genera una sobrecarga de aproximadamente 3x al emplear este nivel de resiliencia.

En el nivel 2, el método TREX voltea aleatoriamente los qubits mediante puertas X inmediatamente antes de la medición,
y voltea el bit medido correspondiente si se aplicó una puerta X. Este enfoque se describe con más detalle en [Model-free
readout-error mitigation for quantum expectation
values](https://arxiv.org/abs/2012.09738).

</details>

### Ejemplo
La interfaz `EstimatorV2` permite a los usuarios trabajar sin complicaciones con la variedad de
métodos de mitigación de errores para reducir el error en los valores esperados de
los observables. El siguiente código usa Extrapolación de Ruido Cero y mitigación de errores de lectura simplemente
configurando `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Configuración personalizada de errores
Puedes activar y desactivar métodos individuales de mitigación y supresión de errores, incluyendo el desacoplamiento dinámico, el twirling de puertas y de medición, la mitigación de errores de medición, PEC y ZNE. Consulta [Técnicas de mitigación y supresión de errores](error-mitigation-and-suppression-techniques) para obtener una explicación de cada uno.

> **Note:** - No todas las opciones están disponibles para ambos primitivos. Consulta la [tabla de opciones disponibles](runtime-options-overview#options-table) para ver la lista de opciones disponibles.
> - No todos los métodos funcionan juntos en todos los tipos de circuitos. Consulta la [tabla de compatibilidad de características](runtime-options-overview#options-compatibility-table) para más detalles.

<Tabs>
  <TabItem value="Estimator" label="Estimator">
    ```python
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}")
print(f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}")
```
  </TabItem>

  <TabItem value="Sampler" label="Sampler">